# LightGBM + early stopping: dự đoán review xấu của Olist

Bài toán giống module 10 và 11: với mỗi đơn **đã giao xong**, dự đoán khách có để lại review 1–2
sao không (`bad_review`), tại thời điểm đơn vừa giao. Chia theo **ngày mua**: train trước
2018-01-01, validation 2018-01 tới 2018-03, test 2018-04 tới 2018-08.

Phần dựng feature (`build_dataset`) và chia tập (`time_split`) đã viết sẵn, giống module 10. Việc
của bạn là phần boosting: early stopping đúng cách, đánh giá đúng số cây, và quan hệ giữa
learning rate với số cây.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật và gọi hàm để xem; test bỏ qua.
- Chấm: `uv run pytest tracks/data-science/modules/12-advanced-ml/exercises/intermediate/01-lightgbm-early-stopping -q`

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score

TRAIN_END = pd.Timestamp("2018-01-01")
VAL_END = pd.Timestamp("2018-04-01")
TEST_END = pd.Timestamp("2018-09-01")

## 0. Dữ liệu (đã viết sẵn)

Mọi feature đều biết được tại lúc đơn được giao: giá, phí ship, số món, số seller, cách thanh
toán, bang của khách/seller, danh mục sản phẩm, và các khoảng thời gian (giao mất bao lâu, hứa bao
lâu, trễ/sớm bao nhiêu ngày so với hẹn). Bốn cột phân loại để kiểu `category`: LightGBM xử lý trực
tiếp, không cần one-hot.

In [ ]:
CATEGORICAL = ["product_category_name", "seller_state", "customer_state", "payment_type"]
FEATURES = ["n_items", "n_sellers", "n_products", "price", "freight", "weight", "installments",
            "n_payments", "delivery_days", "promised_days", "days_vs_estimate", "is_late",
            "approve_hours", "carrier_days", "freight_ratio", "same_state", "purchase_dow"] + CATEGORICAL


def build_dataset(raw_dir) -> pd.DataFrame:
    """Một dòng mỗi đơn giao xong có review: FEATURES + bad_review + order_purchase_timestamp."""
    from pathlib import Path
    raw = Path(raw_dir)
    dates = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
    orders = pd.read_csv(raw / "olist_orders_dataset.csv", parse_dates=dates)
    reviews = pd.read_csv(raw / "olist_order_reviews_dataset.csv", parse_dates=["review_answer_timestamp"])
    items = pd.read_csv(raw / "olist_order_items_dataset.csv")
    payments = pd.read_csv(raw / "olist_order_payments_dataset.csv")
    customers = pd.read_csv(raw / "olist_customers_dataset.csv")
    sellers = pd.read_csv(raw / "olist_sellers_dataset.csv")
    products = pd.read_csv(raw / "olist_products_dataset.csv")

    df = orders[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna()]
    latest = reviews.sort_values("review_answer_timestamp", kind="stable").drop_duplicates("order_id", keep="last")
    df = df.merge(latest[["order_id", "review_score"]], on="order_id")
    df["bad_review"] = (df.review_score <= 2).astype(int)

    it = (items.merge(products[["product_id", "product_category_name", "product_weight_g"]], on="product_id", how="left")
               .merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left"))
    per_order = it.groupby("order_id").agg(
        n_items=("order_item_id", "size"), n_sellers=("seller_id", "nunique"), n_products=("product_id", "nunique"),
        price=("price", "sum"), freight=("freight_value", "sum"), weight=("product_weight_g", "sum"))
    first_item = (it.sort_values("order_item_id").drop_duplicates("order_id")
                    [["order_id", "product_category_name", "seller_state"]])
    pay = payments.groupby("order_id").agg(installments=("payment_installments", "max"),
                                           n_payments=("payment_sequential", "size"),
                                           payment_type=("payment_type", "first"))
    df = (df.merge(per_order, on="order_id", how="left").merge(first_item, on="order_id", how="left")
            .merge(pay, on="order_id", how="left")
            .merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left"))

    day = pd.Timedelta(days=1)
    df["delivery_days"] = (df.order_delivered_customer_date - df.order_purchase_timestamp) / day
    df["promised_days"] = (df.order_estimated_delivery_date - df.order_purchase_timestamp) / day
    df["days_vs_estimate"] = (df.order_delivered_customer_date - df.order_estimated_delivery_date) / day
    df["is_late"] = (df.order_delivered_customer_date > df.order_estimated_delivery_date).astype(int)
    df["approve_hours"] = (df.order_approved_at - df.order_purchase_timestamp) / pd.Timedelta(hours=1)
    df["carrier_days"] = (df.order_delivered_carrier_date - df.order_purchase_timestamp) / day
    df["freight_ratio"] = df.freight / df.price
    df["same_state"] = (df.customer_state == df.seller_state).astype(int)
    df["purchase_dow"] = df.order_purchase_timestamp.dt.day_of_week
    for col in CATEGORICAL:
        df[col] = df[col].astype("category")
    return df[["order_id", "order_purchase_timestamp"] + FEATURES + ["bad_review"]].reset_index(drop=True)


def time_split(df: pd.DataFrame, date_col: str = "order_purchase_timestamp") -> dict[str, pd.DataFrame]:
    """{"train", "val", "test"} theo ngày mua; bỏ đơn mua từ TEST_END trở đi."""
    t = df[date_col]
    return {"train": df[t < TRAIN_END], "val": df[(t >= TRAIN_END) & (t < VAL_END)],
            "test": df[(t >= VAL_END) & (t < TEST_END)]}

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
data = build_dataset(ROOT / "data" / "raw")
splits = time_split(data)
pd.DataFrame({name: {"n": len(part), "bad_rate": round(part.bad_review.mean(), 4)} for name, part in splits.items()}).T

In [ ]:
# explore: vì sao validation có tỉ lệ review xấu cao hơn hẳn train và test? Xem theo tháng.
data.assign(month=data.order_purchase_timestamp.dt.to_period("M")).groupby("month").agg(
    n=("bad_review", "size"), bad_rate=("bad_review", "mean"), late_rate=("is_late", "mean")).loc["2017-06":"2018-08"].round(3)

## 1. Early stopping

`PARAMS` là bộ tham số khởi điểm hợp lý (giải thích trong mục 3 của lý thuyết). Viết
`fit_early_stopping`:

- `lgb.train(params, train_set, num_boost_round=max_rounds, valid_sets=[val_set], callbacks=[lgb.early_stopping(patience, verbose=False)])`
- Theo dõi **`binary_logloss` trên validation** (đặt `metric` trong params). Không bao giờ đưa test
  vào `valid_sets`.
- `params=None` nghĩa là dùng `PARAMS`. Không sửa dict người gọi truyền vào.

`evaluate(booster, X, y)` trả `{"auc", "pr_auc", "logloss"}`, dự đoán bằng **`best_iteration`**.

In [ ]:
PARAMS = dict(objective="binary", metric="binary_logloss", learning_rate=0.05, num_leaves=31,
              min_data_in_leaf=50, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
              lambda_l2=1.0, num_threads=1, verbose=-1, seed=0)

In [ ]:
def fit_early_stopping(X_train, y_train, X_val, y_val, params=None, max_rounds=2000, patience=50) -> lgb.Booster:
    """lgb.train với early stopping theo binary_logloss trên (X_val, y_val). Trả Booster có best_iteration."""
    params = dict(PARAMS if params is None else params)
    params["metric"] = "binary_logloss"
    train_set = lgb.Dataset(X_train, y_train)
    val_set = lgb.Dataset(X_val, y_val, reference=train_set)
    return lgb.train(params, train_set, num_boost_round=max_rounds, valid_sets=[val_set],
                     callbacks=[lgb.early_stopping(patience, verbose=False)])


def evaluate(booster: lgb.Booster, X, y) -> dict[str, float]:
    """{"auc", "pr_auc", "logloss"} với dự đoán tại booster.best_iteration."""
    p = booster.predict(X, num_iteration=booster.best_iteration)
    return {"auc": float(roc_auc_score(y, p)), "pr_auc": float(average_precision_score(y, p)),
            "logloss": float(log_loss(y, p))}

In [ ]:
# explore: train với early stopping, so với 1000 vòng không dừng
tr, va, te = splits["train"], splits["val"], splits["test"]
model = fit_early_stopping(tr[FEATURES], tr.bad_review, va[FEATURES], va.bad_review)
print("best_iteration:", model.best_iteration)
print("val ", evaluate(model, va[FEATURES], va.bad_review))
print("test", evaluate(model, te[FEATURES], te.bad_review))

# cùng tham số, không dừng sớm: train đủ 1000 cây, ghi lại logloss validation từng vòng
history = {}
train_set = lgb.Dataset(tr[FEATURES], tr.bad_review)
no_es = lgb.train(PARAMS, train_set, num_boost_round=1000,
                  valid_sets=[lgb.Dataset(va[FEATURES], va.bad_review, reference=train_set)],
                  callbacks=[lgb.record_evaluation(history)])
p = no_es.predict(te[FEATURES])
print("1000 vòng, test AUC", round(roc_auc_score(te.bad_review, p), 4), "logloss", round(log_loss(te.bad_review, p), 4))

In [ ]:
# explore: đường cong logloss validation theo số cây
import matplotlib.pyplot as plt

curve = history["valid_0"]["binary_logloss"]
plt.plot(range(1, len(curve) + 1), curve)
plt.axvline(model.best_iteration, ls="--", c="gray")
plt.xlabel("số cây"); plt.ylabel("logloss validation"); plt.show()

## 2. Learning rate và số cây

`lr_sweep` chạy `fit_early_stopping` cho từng learning rate (giữ nguyên các tham số khác) và trả
DataFrame index `learning_rate` (đúng thứ tự `rates`), cột `best_iteration`, `val_logloss`
(logloss validation tại best_iteration), `test_auc`.

In [ ]:
def lr_sweep(X_train, y_train, X_val, y_val, X_test, y_test, rates=(0.2, 0.1, 0.05, 0.02), params=None) -> pd.DataFrame:
    """Một dòng mỗi learning rate: best_iteration, val_logloss, test_auc."""
    base = dict(PARAMS if params is None else params)
    rows = []
    for rate in rates:
        booster = fit_early_stopping(X_train, y_train, X_val, y_val, params={**base, "learning_rate": rate})
        rows.append({"learning_rate": rate, "best_iteration": booster.best_iteration,
                     "val_logloss": evaluate(booster, X_val, y_val)["logloss"],
                     "test_auc": evaluate(booster, X_test, y_test)["auc"]})
    return pd.DataFrame(rows).set_index("learning_rate")

In [ ]:
# explore
lr_sweep(tr[FEATURES], tr.bad_review, va[FEATURES], va.bad_review, te[FEATURES], te.bad_review).round(4)

In [ ]:
# explore: feature nào đóng góp nhiều nhất (gain)?
imp = pd.Series(model.feature_importance("gain"), index=FEATURES)
(imp / imp.sum()).sort_values(ascending=False).head(10).round(3)

## 3. Câu hỏi (viết câu trả lời vào cell này)

1. AUC validation cao hơn AUC test khá nhiều. Cell explore thứ hai gợi ý vì sao. Điều đó nói gì
   về việc dùng chính tập validation này để chọn số cây?
2. Sau khi chọn được `best_iteration`, bạn có train lại trên train + validation không? Nếu có,
   dùng bao nhiêu cây?
3. `days_vs_estimate` chiếm phần lớn gain. Nếu muốn dùng model **trước khi** đơn được giao (để can
   thiệp sớm), feature nào phải bỏ, và bạn kỳ vọng AUC thay đổi ra sao?

**Trả lời (lời giải mẫu)**

1. Quý 1/2018 là giai đoạn giao trễ nặng (tỉ lệ trễ 15.9% tháng 2, 21.1% tháng 3, so với 3–8% ở
   các tháng khác), nên review xấu lên 18% và "trễ → review xấu" dễ phân biệt hơn: AUC validation
   0.820, test 0.707. Validation vẫn hợp lệ để chọn số cây (nó nằm sau train, trước test), nhưng
   con số AUC của nó **không** phải ước lượng cho production. Báo cáo hiệu năng bằng test, và nếu
   được, chọn số cây trên nhiều cửa sổ thời gian (rolling origin, module 10).
2. Có, vì dữ liệu gần nhất có giá trị nhất. Train lại trên train + validation với 117 cây cho test
   AUC 0.714 (so với 0.707); nhân số cây theo tỉ lệ dữ liệu (172 cây) cho 0.712. Chênh lệch nhỏ;
   điều quan trọng là **không** dùng test để chọn số cây.
3. Can thiệp trước khi giao là một **điểm dự đoán khác** (lúc mua hoặc lúc giao cho hãng vận
   chuyển), không phải "model này bỏ bớt cột": mọi kết quả giao hàng sau thời điểm đó đều là
   leakage. Tối thiểu phải bỏ `delivery_days`, `days_vs_estimate`, `is_late` (và `carrier_days` nếu
   dự đoán lúc mua). Chỉ bỏ ba cột đầu, test AUC giảm từ
   0.707 xuống 0.656, PR-AUC từ 0.347 xuống 0.238. Model sớm yếu hơn nhưng có thời gian để hành
   động; model muộn chính xác hơn nhưng chỉ còn cách xin lỗi. Chọn theo **quyết định** cần hỗ trợ.